# Random Forest classification

GridSearchCV over a RandomForestClassifier with a notebook progress bar using `tqdm_joblib`.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Create output directory if it doesn't exist
os.makedirs("data/2_training", exist_ok=True)

# Load preprocessed data
df = pd.read_csv("data/1_preprocessed/preprocessed_dataset.csv")

# Create train/test/val split (70% train, 15% test, 15% val)
# First split: 70% train, 30% temp (which will be split into test and val)
X_train, X_temp, y_train, y_temp = train_test_split(
    df, df, test_size=0.3, random_state=42, stratify=df["track_genre"]
)

# Second split: split temp into 50% test and 50% val (each 15% of original)
X_test, X_val, y_test, y_val = train_test_split(
    X_temp, X_temp, test_size=0.5, random_state=42, stratify=X_temp["track_genre"]
)

# Save as CSVs
train_df = X_train[0]  # Get the DataFrame from the split
test_df = X_test[0]
val_df = X_val[0]

train_df.to_csv("data/2_training/train.csv", index=False)
test_df.to_csv("data/2_training/test.csv", index=False)
val_df.to_csv("data/2_training/val.csv", index=False)

print(f"Train set: {len(train_df)} samples ({len(train_df)/len(df)*100:.1f}%)")
print(f"Test set: {len(test_df)} samples ({len(test_df)/len(df)*100:.1f}%)")
print(f"Validation set: {len(val_df)} samples ({len(val_df)/len(df)*100:.1f}%)")
print("\nFiles saved to data/2_training/")
print("- train.csv")
print("- test.csv")
print("- val.csv")

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv("data/1_preprocessed/preprocessed_dataset.csv")
X = df.drop(columns=["track_genre","track_id","artists","album_name","track_name"])
y = df["track_genre"]

numeric_cols = ["danceability","energy","loudness","speechiness","acousticness",
                "instrumentalness","liveness","valence","tempo","duration_min"]
cat_cols = ["explicit","mode","key","time_signature"]

preproc = ColumnTransformer([
    ("num", "passthrough", numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

pipe = Pipeline([("pre", preproc), ("clf", RandomForestClassifier(random_state=42))])

param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [None, 10, 20],
    'clf__min_samples_leaf': [1, 2],
    'clf__max_features': ['sqrt', 'log2']
}

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)
grid = GridSearchCV(pipe, param_grid, cv=5, n_jobs=-1)

# Progress: total number of fits = n_candidates * cv
cv = 5
n_candidates = len(list(ParameterGrid(param_grid)))
total_fits = n_candidates * cv
with tqdm_joblib(tqdm(desc="GridSearch fits", total=total_fits)):
    grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
y_pred = grid.predict(X_test)
print(classification_report(y_test, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

GridSearch fits:   0%|          | 0/120 [00:00<?, ?it/s]

  0%|          | 0/120 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [5]:
# Plot top feature importances
best = grid.best_estimator_.named_steps['clf']
# get feature names after ColumnTransformer
onehot = grid.best_estimator_.named_steps['pre'].named_transformers_['cat']
try:
    oh_names = list(onehot.get_feature_names_out(cat_cols))
except Exception:
    oh_names = []
feature_names = numeric_cols + oh_names
importances = best.feature_importances_
idx = importances.argsort()[::-1][:20]
plt.figure(figsize=(8,6))
plt.barh([feature_names[i] for i in idx][::-1], importances[idx][::-1])
plt.title('Top 20 feature importances')
plt.tight_layout()
plt.show()

AttributeError: 'GridSearchCV' object has no attribute 'best_estimator_'

In [ ]:
# Validation metrics on completely unseen data
val_df = pd.read_csv("data/2_training/val.csv")
X_val = val_df.drop(columns=["track_genre","track_id","artists","album_name","track_name"])
y_val = val_df["track_genre"]

y_val_pred = grid.predict(X_val)

print("=== VALIDATION SET RESULTS ===")
print(f"Validation set size: {len(y_val)} samples\n")
print("Classification Report:")
print(classification_report(y_val, y_val_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))